# Relatório de Modificações — smosaic

**Objetivo:** Inclusão de novos métodos de composição temporal baseados em xarray, adaptados do notebook `Pipeline_DataCube_novas_funcoes_v2.ipynb`, na pipeline existente do smosaic.


---

## Sumário

| # | Arquivo | Tipo de Mudança |
|---|---|---|
| 1 | `smosaic_new_methods.py` | Novo arquivo — núcleo dos novos métodos |
| 2 | `smosaic_mosaic.py` | Refatoração da função `process_period` |
| 3 | `cli.py` | Novos parâmetros de linha de comando |
| 4 | `setup.py` | Nova dependência `xarray` |
| 5 | `smosaic_merge_tifs.py` | Correção de bug — nodata hardcoded |
| 6 | `smosaic_fix_baseline_number.py` | Correção de bug — NaN inválido para int16 |
| 7 | `tests/` | Novos testes (Níveis 1, 2 e 3) para os novos métodos |
| 8 | `tests/` | Testes de regressão para os módulos antigos |
| 9 | — | Instruções para executar todos os testes |
| 10 | `smosaic_new_methods.py`, `smosaic_mosaic.py` | Suporte a bandas derivadas via `COMPUTED_BANDS` |
| 11 | `output_*/` | Execuções reais com dados STAC BDC — bbox MT, ago/2024 |

---
## 1. `smosaic_new_methods.py` — Novo arquivo

### Motivo
O trabalho para introduzir seis novos métodos de composição temporal que operam sobre cubos xarray `(time, y, x)`. Para manter o código organizado e separar a lógica dos novos métodos da pipeline principal, foi criado um arquivo dedicado.

### Métodos implementados

| Método CLI | Alias | Descrição | Proveniência |
|---|---|---|---|
| `avg` | `media` | Média temporal por pixel | Nenhuma |
| `med` | `mediana` | Mediana temporal por pixel | Nenhuma |
| `max` | — | Máximo por banda, por pixel | Por banda (multi-banda) |
| `min` | — | Mínimo por banda, por pixel | Por banda (multi-banda) |
| `maxx` | — | k-ésimo máximo da banda de referência | Banda única (ref) |
| `minx` | — | k-ésimo mínimo da banda de referência | Banda única (ref) |

### Constante de roteamento

In [ ]:
NEW_METHODS = {'avg', 'media', 'med', 'mediana', 'max', 'min', 'maxx', 'minx'}

### `COMPUTED_BANDS` — bandas derivadas

Mapeia nomes de bandas **derivadas** (como `NBR`) para as bandas reais que precisam ser baixadas do STAC. Permite ao usuário solicitar `NBR` como banda de saída sem que ela precise existir em disco — o pipeline baixa `B08` e `B12` automaticamente e calcula o índice em memória.

Usada em dois pontos:
- `mosaic()` — na verificação/download das bandas antes do processamento.
- `process_period()` — na Fase 1, para construir `phase1_bands` (bandas reais a reprojetar) a partir de `bands` (bandas solicitadas, que podem incluir `NBR`).

In [ ]:
COMPUTED_BANDS = {'NBR': ('B08', 'B12')}

# Expansão em process_period() — Fase 1
phase1_bands = []
for b in bands:
    if b in COMPUTED_BANDS:
        for rb in COMPUTED_BANDS[b]:   # 'NBR' -> ['B08', 'B12']
            if rb not in phase1_bands:
                phase1_bands.append(rb)
    else:
        phase1_bands.append(b)
# resultado: bands=['NBR'] -> phase1_bands=['B08','B12'] (baixa do STAC)
#            bands=['B04','NBR'] -> phase1_bands=['B04','B08','B12']

### `calcular_nbr_cubo`
Adiciona `NBR = (NIR - SWIR) / (NIR + SWIR + ε)` ao Dataset sem mutação. Necessária quando `banda_ref='NBR'` nos métodos `maxx`/`minx`.

In [ ]:
def calcular_nbr_cubo(cubo, nir_band='B08', swir_band='B12', eps=1e-7):
    nir  = cubo[nir_band].astype(np.float32)
    swir = cubo[swir_band].astype(np.float32)
    nbr  = (nir - swir) / (nir + swir + eps)
    return cubo.assign(NBR=nbr)

### `_build_xarray_cube`
Constrói o Dataset `(time, y, x)` a partir dos TIFs reprojetados, aplicando máscara de nuvem. 

**Bug identificado:** ao usar bandas de resoluções diferentes (B04/B08 em 10m e B12 em 20m), a leitura direta com `src.read(1)` retornava arrays com dimensões incompatíveis com o cloud_mask, causando `IndexError`.

**Correção:** todas as bandas são reamostradas para a resolução da banda de referência (primeira da lista) usando `out_shape=(height, width)`.

In [ ]:
# ANTES — lia na resolução nativa, causando IndexError com bandas mistas
# with rasterio.open(band_file) as src:
#     arr = src.read(1).astype(np.float32)  # B12: (5490,5490), mas height=10981 (de B04)

# DEPOIS — resample para a resolução da banda de referência
with rasterio.open(band_file) as src:
    arr = src.read(
        1,
        out_shape=(height, width),       # height/width vêm de bands[0]
        resampling=Resampling.nearest,
    ).astype(np.float32)

---
## 2. `smosaic_mosaic.py` — Refatoração de `process_period`

### Motivo
A função original processava uma banda por vez em loop único. Os novos métodos (ex.: `maxx` com `banda_ref='NBR'`) precisam de **todas as bandas simultaneamente** para calcular índices espectrais antes da composição. A solução foi reestruturar em 3 fases independentes.

### Estrutura em 3 fases

```
┌─────────────────────────────────────────────────────────┐
│  PHASE 1 — Download → Filter → Sort → Reproject         │
│  Loop por banda → all_sorted_data{band: [items...]}     │
│  (igual para TODOS os métodos)                          │
└──────────────────────────┬──────────────────────────────┘
                           │
           ┌───────────────┴───────────────┐
           │ mosaic_method in NEW_METHODS? │
           └───────┬───────────────┬───────┘
                  SIM             NÃO
                   │               │
    ┌──────────────▼──┐    ┌───────▼──────────────┐
    │ compose_new_    │    │ merge_scene /        │
    │ method(...)     │    │ merge_scene_prov...()│
    │ (xarray-based)  │    │ (lógica original)    │
    └──────────────┬──┘    └───────┬──────────────┘
                   └───────┬───────┘
                           │ composition_results{band: {...}}
┌──────────────────────────▼──────────────────────────────┐
│  PHASE 3 — merge_tifs → clip_raster →                   │
│            fix_baseline → generate_cog → clean_dir      │
│  (igual para TODOS os métodos)                          │
└─────────────────────────────────────────────────────────┘
```

### Mudanças na assinatura de `mosaic()` e `process_period()`

In [ ]:
# Parâmetros adicionados em mosaic() e process_period()
# k        — rank para maxx/minx (1 = extremo absoluto)
# banda_ref — banda de referência para maxx/minx (ex.: 'B12' ou 'NBR')

def mosaic(..., mosaic_method, ..., k=1, banda_ref=None):
    ...
    args_for_processes = [
        (period, mosaic_method, ..., k, banda_ref)   # k e banda_ref adicionados ao tuple
        for period in periods
    ]

def process_period(period, mosaic_method, ..., k=1, banda_ref=None):
    ...

### Import adicionado

In [ ]:
from smosaic.smosaic_new_methods import NEW_METHODS, compose_new_method, COMPUTED_BANDS

---
## 3. `cli.py` — Novos parâmetros de linha de comando

### Motivo
Expor os novos métodos e seus parâmetros específicos na interface CLI.

### Mudanças

In [ ]:
# ANTES
@click.option('--mosaic-method',
              type=click.Choice(['lcf', 'chrono', 'ctd'], case_sensitive=False),
              required=True)

# DEPOIS
@click.option('--mosaic-method',
              type=click.Choice(
                  ['lcf', 'chrono', 'ctd',
                   'avg', 'media', 'med', 'mediana',
                   'max', 'min', 'maxx', 'minx'],
                  case_sensitive=False,
              ),
              required=True)

# ADICIONADOS
@click.option('--k', type=int, default=1, show_default=True,
              help='Rank para métodos maxx/minx (1 = extremo absoluto)')
@click.option('--banda-ref', default=None,
              help='Banda de referência para maxx/minx (ex.: B12 ou NBR)')

### Exemplos de uso via CLI

In [ ]:
exemplos = {
    "avg": "smosaic mosaic --collection S2_L2A-1 --mosaic-method avg "
           "--band B04 --band B08 --bbox \"-54.5,-12.5,-54.0,-12.0\" "
           "--start-year 2024 --start-month 6 --start-day 1 --duration-days 30 --output-dir output/",

    "maxx (B12)": "smosaic mosaic --collection S2_L2A-1 --mosaic-method maxx "
                  "--band B04 --band B08 --band B12 --bbox \"-54.5,-12.5,-54.0,-12.0\" "
                  "--start-year 2024 --start-month 6 --start-day 1 --duration-days 30 "
                  "--k 1 --banda-ref B12 --output-dir output/",

    "maxx (NBR)": "smosaic mosaic --collection S2_L2A-1 --mosaic-method maxx "
                  "--band B04 --band B08 --band B12 --bbox \"-54.5,-12.5,-54.0,-12.0\" "
                  "--start-year 2024 --start-month 6 --start-day 1 --duration-days 30 "
                  "--k 1 --banda-ref NBR --output-dir output/",
}

for metodo, cmd in exemplos.items():
    print(f"[{metodo}]\n{cmd}\n")

---
## 4. `setup.py` — Nova dependência

In [ ]:
install_requires = [
    "numpy==2.3.4",
    "tqdm==4.67.1",
    "pyproj==3.7.2",
    "shapely==2.1.2",
    "requests==2.32.5",
    "rasterio==1.4.3",
    "pystac-client==0.9.0",
    "xarray>=2024.1.0",   # ADICIONADO — necessário para smosaic_new_methods.py
]

---
## 5. `smosaic_merge_tifs.py`

### Problema
`nodata = 0` estava hardcoded para bandas espectrais. Os novos métodos escrevem `nodata = NaN` (float32). Em mosaicos multi-tile, o GDAL merge com `-n 0` não reconhecia pixels NaN como transparentes, causando buracos no resultado final.

### Correção

In [ ]:
# ANTES
# nodata = 0   # hardcoded

# DEPOIS
src_ds = gdal.Open(tif_files[0])
dt = src_ds.GetRasterBand(1).DataType
dtype_name = gdal.GetDataTypeName(dt)
file_nodata = src_ds.GetRasterBand(1).GetNoDataValue()   # lê do arquivo
src_ds = None

cloud_dict = get_all_cloud_configs()
if any(config['cloud_band'] == band for config in cloud_dict.values()):
    nodata = next((config['no_data_value'] for config in cloud_dict.values()
                   if config['cloud_band'] == band), None)
else:
    nodata = file_nodata if file_nodata is not None else 0

| Cenário | nodata no arquivo | Antes | Depois |
|---|---|---|---|
| Novos métodos — banda espectral | `NaN` | `0` ❌ | `NaN` ✅ |
| Métodos antigos — banda espectral | `0` | `0` ✅ | `0` ✅ |
| Qualquer método — banda cloud | qualquer | cloud_dict ✅ | cloud_dict ✅ |
| Qualquer método — proveniência | `0` (int32) | `0` ✅ | `0` ✅ |

---
## 6. `smosaic_fix_baseline_number.py`

### Problema
Para `baseline_number > 400`, a função força `dtype='int16'` sem atualizar o `nodata`. Com arquivos dos novos métodos (`nodata=NaN`), o rasterio lançava:

```
ValueError: Given nodata value, nan, is beyond the valid range of its data type, int16.
```

### Correção

In [ ]:
# ANTES
# new_image_data = image_data.astype('int16') - 1000
# profile.update({'dtype': 'int16'})   # nodata=NaN permanecia → ValueError

# DEPOIS
current_nodata = profile.get('nodata')
has_nan_nodata = (
    current_nodata is not None
    and isinstance(current_nodata, float)
    and math.isnan(current_nodata)
)

if has_nan_nodata:
    nan_mask = np.isnan(image_data)
    new_image_data = np.where(nan_mask, np.float32(0), image_data - 1000).astype('int16')
    new_nodata = 0
else:
    new_image_data = image_data.astype('int16') - 1000
    new_nodata = current_nodata

profile.update({'dtype': 'int16', 'nodata': new_nodata})   # nodata compatível com int16

---
## 7. `tests/` — Novos testes para os novos métodos

### Nível 1 — Funções puras (`test_new_methods.py`)
Cubos xarray sintéticos, sem I/O.

In [ ]:
testes_nivel_1 = [
    "test_proveniencia_dia_do_ano             — formato 1-366 correto (1 jan, 15 jun, 31 dez)",
    "test_proveniencia_nodata_zero_mascarados — pixels mascarados retornam 0",
    "test_media_ignora_nan                    — avg ignora NaN no cálculo da média",
    "test_max_retorna_valor_maximo_por_banda  — max seleciona o maior valor",
    "test_min_retorna_valor_minimo_por_banda  — min seleciona o menor valor",
    "test_maxx_usa_banda_referencia           — maxx usa banda de referência para escolher data",
    "test_nbr_calculo                         — NBR = (NIR-SWIR)/(NIR+SWIR+ε) ≈ 0.333",
    "test_aplicar_metodo_routing              — avg/med retornam prov=None",
]
for t in testes_nivel_1:
    print(t)

### Nível 2 — TIFs sintéticos (`test_merge_tifs.py`)
Cria arquivos TIF temporários para verificar o comportamento do `merge_tifs` corrigido.

In [ ]:
testes_nivel_2 = [
    "test_nan_nodata_preenche_tiles_adjacentes  — bug corrigido: tiles NaN mesclam sem buracos",
    "test_nan_nodata_tile_unico_preserva        — tile único com NaN preserva valores válidos",
    "test_nodata_zero_preenche_tiles_adjacentes — regressão: métodos antigos (nodata=0) OK",
    "test_cloud_band_scl_usa_nodata_cloud_dict  — SCL usa nodata do cloud_dict (0)",
    "test_proveniencia_int32_dois_tiles         — proveniência int32 mescla corretamente",
]
for t in testes_nivel_2:
    print(t)

### Nível 3 — End-to-end com dados reais do STAC BDC
Pipeline completa com download real. Divididos em 3 arquivos para controle de memória RAM.

| Arquivo | Método | Bandas | COGs gerados | Resultado |
|---|---|---|---|---|
| `test_e2e_avg.py` | `avg` | B04 | B04 + SCL | ✅ PASS |
| `test_e2e_max.py` | `max` | B04, B08 | B04 + B08 + PROVENANCE + SCL | ✅ PASS |
| `test_e2e_maxx.py` | `maxx` | B04, B08, B12 | B04 + B08 + B12 + PROVENANCE + SCL | ✅ PASS |

In [ ]:
# Como executar cada parte separadamente
comandos = [
    "pytest smosaic/tests/test_e2e_avg.py  -v -s  # Parte 1 — avg  (1 banda)",
    "pytest smosaic/tests/test_e2e_max.py  -v -s  # Parte 2 — max  (2 bandas)",
    "pytest smosaic/tests/test_e2e_maxx.py -v -s  # Parte 3 — maxx (3 bandas)",
]
for cmd in comandos:
    print(cmd)

---
## 8. `tests/` — Testes de regressão para os módulos antigos

### Motivo
As novas implementações (`smosaic_new_methods.py`, refatoração do `process_period`) poderiam introduzir regressões silenciosas nos módulos que não foram diretamente modificados. Para garantir que o comportamento original se mantém, foram criados testes unitários e end-to-end cobrindo todos os módulos antigos sem cobertura prévia.

### Módulos cobertos

| Arquivo de teste | Módulo testado | Nível | Nº de testes |
|---|---|---|---|
| `test_utils.py` | `smosaic_utils` | 1 | 14 |
| `test_count_pixels.py` | `smosaic_count_pixels` | 2 | 4 |
| `test_fix_baseline_number.py` | `smosaic_fix_baseline_number` | 2 | 4 |
| `test_spectral_indices.py` | `smosaic_spectral_indices` | 2 | 6 |
| `test_clip_raster.py` | `smosaic_clip_raster` | 2 | 3 |
| `test_generate_cog.py` | `smosaic_generate_cog` | 2 | 4 |
| `test_e2e_old_methods.py` | pipeline completa (`lcf`, `chrono`, `ctd`) | 3 | 3 |

### 8.1 Nível 1 — Utilitários (`test_utils.py`)
Funções puras sem I/O de disco. Sem dependências externas.

In [ ]:
testes_utils = [
    "test_cloud_config_colecoes_presentes          — S2_L2A-1 e S2_L1C_BUNDLE-1 existem no dicionário",
    "test_cloud_config_s2l2a1_banda_e_nodata       — banda=SCL, nodata=0, non_cloud=[4,5,6]",
    "test_cloud_config_s2l1c_banda_e_nodata        — banda=FMASK, nodata=255, non_cloud=[0,1]",
    "test_cloud_config_retorna_copia_independente  — modificar retorno não altera o config original",
    "test_add_months_to_date_retorna_ultimo_dia    — jan+1 mês → 29/fev (2024 bissexto)",
    "test_add_months_to_date_aceita_datetime       — aceita datetime além de string",
    "test_days_between_dates_diferenca_positiva    — diferença de 5 dias entre as datas",
    "test_days_between_dates_mesmo_dia             — mesma data → 0 dias",
    "test_add_days_to_date_soma_dias               — adiciona 10 dias corretamente",
    "test_add_days_to_date_vira_mes                — virada de mês calculada corretamente",
    "test_geometry_intersecta_bbox                 — geometria que intersecta o bbox retorna True",
    "test_geometry_nao_intersecta_bbox             — geometria fora do bbox retorna False",
    "test_clean_dir_sem_args_remove_tifs_nao_cog   — remove .tif mas preserva _COG.tif e outros",
    "test_create_composition_json_cria_arquivo     — JSON com collection, scenes e ignored_scenes",
]
for t in testes_utils:
    print(t)

### 8.2 Nível 2 — TIFs sintéticos (módulos individuais)
Cada teste cria rasters temporários com `tempfile` + `rasterio`, sem acesso à rede.

In [ ]:
testes_nivel_2_modulos_antigos = {
    "test_count_pixels.py": [
        "test_count_pixels_identifica_valores_alvo    — todos SCL=4 → count == total (16)",
        "test_count_pixels_nuvem_nao_e_alvo           — metade nuvem → count = 8, total = 16",
        "test_count_pixels_nodata_excluido_do_total   — pixels nodata=0 não entram no total",
        "test_count_pixels_retorna_dict_com_chaves    — retorno tem 'total' e 'count' como int",
    ],
    "test_fix_baseline_number.py": [
        "test_baseline_abaixo_400_nao_altera_dados    — baseline 399 → sem mudança nos valores",
        "test_baseline_igual_a_400_nao_altera_dados   — baseline 400 → sem mudança nos valores",
        "test_baseline_acima_400_subtrai_1000         — baseline 401 → todos os valores − 1000",
        "test_baseline_acima_400_nan_nodata           — NaN vira 0, restante − 1000, nodata→0",
    ],
    "test_spectral_indices.py": [
        "test_fix_negatives_zera_valores_negativos    — valores < 0 viram 0",
        "test_fix_negatives_preserva_positivos        — valores ≥ 0 não são alterados",
        "test_ndvi_formula_correta                    — (1000−500)/(1000+500)×10000 = 3333",
        "test_ndvi_output_dtype_int16                 — arquivo de saída em int16",
        "test_evi2_formula_correta                    — 2.5×500/1501×10000 ≈ 8328",
        "test_savi_formula_correta                    — 1.5×500/1500.5×10000 ≈ 4998",
    ],
    "test_clip_raster.py": [
        "test_clip_raster_recorta_area_menor          — output menor que o raster original",
        "test_clip_raster_retorna_path_correto        — retorna o path esperado e deleta o input",
        "test_clip_raster_preserva_valores_validos    — pixels do AOI têm valor original (42.0)",
    ],
    "test_generate_cog.py": [
        "test_generate_cog_cria_arquivo_cog           — arquivo _COG.tif criado em disco",
        "test_generate_cog_retorna_path_correto       — retorno == pasta/nome_COG.tif",
        "test_generate_cog_arquivo_e_raster_valido    — COG tem ≥1 banda e pixels não-nulos",
        "test_generate_cog_output_dtype_int16         — COG em int16 (conversão documentada)",
    ],
}

for arquivo, testes in testes_nivel_2_modulos_antigos.items():
    print(f"\n[{arquivo}]")
    for t in testes:
        print(f"  {t}")

### 8.3 Nível 3 — End-to-end para métodos antigos (`test_e2e_old_methods.py`)

Pipeline completa com download real do STAC BDC. Verifica que `lcf`, `chrono` e `ctd` continuam gerando a estrutura de saída correta após as novas implementações.

| Teste | Método | Bandas | Saída esperada | Resultado |
|---|---|---|---|---|
| `test_e2e_lcf` | `lcf` — menor nuvem primeiro | B04 | B04 COG + PROVENANCE COG + SCL COG | ✅ PASS |
| `test_e2e_chrono` | `chrono` — ordem cronológica | B04, B08 | B04 + B08 + PROVENANCE + SCL COGs | ✅ PASS |
| `test_e2e_ctd` | `ctd` — mais próximo de 2024-06-05 | B04 | B04 COG + PROVENANCE COG + SCL COG | ✅ PASS |

> **Nota:** `chrono` com 2 bandas verifica que `merge_scene_provenance_cloud` (bands[0]) e `merge_scene` (bands[1:]) coexistem corretamente — apenas **1 COG de proveniência** é gerado, não um por banda.

In [ ]:
saida_lcf = [
    "S2_L2A-TEST-10D-B04_20240601_20240610_COG.tif",
    "S2_L2A-TEST-10D_SCL_20240601_20240610_COG.tif",
    "S2_L2A-TEST-10D-PROVENANCE_20240601_20240610_COG.tif",
]

saida_chrono = [
    "S2_L2A-TEST-10D-B04_20240601_20240610_COG.tif",
    "S2_L2A-TEST-10D-B08_20240601_20240610_COG.tif",
    "S2_L2A-TEST-10D_SCL_20240601_20240610_COG.tif",
    "S2_L2A-TEST-10D-PROVENANCE_20240601_20240610_COG.tif",
]

print("lcf / ctd (1 banda):")
for f in saida_lcf:
    print(f"  {f}")

print("\nchrono (2 bandas):")
for f in saida_chrono:
    print(f"  {f}")

---
## 9. Como Executar os Testes

### Pré-requisitos

```bash
# Instalar o pacote em modo editável (necessário para os imports funcionarem)
pip install -e .

# Instalar dependência de teste
pip install pytest
```

---

### 9.1 Todos os testes rápidos (sem rede) — Níveis 1 e 2

Não requerem acesso ao STAC BDC.

```bash
pytest smosaic/tests/test_new_methods.py \
       smosaic/tests/test_merge_tifs.py \
       smosaic/tests/test_utils.py \
       smosaic/tests/test_count_pixels.py \
       smosaic/tests/test_fix_baseline_number.py \
       smosaic/tests/test_spectral_indices.py \
       smosaic/tests/test_clip_raster.py \
       smosaic/tests/test_generate_cog.py \
       -v
```

---

### 9.2 Testes end-to-end — Novos métodos (Nível 3)

Requerem acesso ao servidor STAC BDC (`https://data.inpe.br/bdc/stac/v1`).  
Execute **um por vez** para não sobrecarregar a memória RAM.

```bash
# avg — média temporal (1 banda)
pytest smosaic/tests/test_e2e_avg.py -v -s

# max — máximo por banda (2 bandas)
pytest smosaic/tests/test_e2e_max.py -v -s

# maxx — k-ésimo máximo por banda de referência (3 bandas)
pytest smosaic/tests/test_e2e_maxx.py -v -s
```

---

### 9.3 Testes end-to-end — Métodos antigos (Nível 3)

Requerem acesso ao STAC BDC. Execute individualmente.

```bash
# Todos de uma vez
pytest smosaic/tests/test_e2e_old_methods.py -v -s

# Ou individualmente:
pytest smosaic/tests/test_e2e_old_methods.py::test_e2e_lcf    -v -s
pytest smosaic/tests/test_e2e_old_methods.py::test_e2e_chrono -v -s
pytest smosaic/tests/test_e2e_old_methods.py::test_e2e_ctd    -v -s
```

---

### 9.4 Resumo — todos os testes e seus resultados

| Arquivo | Nível | Requer rede | Nº de testes | Status |
|---|---|---|---|---|
| `test_new_methods.py` | 1 | Não | 8 | ✅ |
| `test_merge_tifs.py` | 2 | Não | 5 | ✅ |
| `test_utils.py` | 1 | Não | 14 | ✅ |
| `test_count_pixels.py` | 2 | Não | 4 | ✅ |
| `test_fix_baseline_number.py` | 2 | Não | 4 | ✅ |
| `test_spectral_indices.py` | 2 | Não | 6 | ✅ |
| `test_clip_raster.py` | 2 | Não | 3 | ✅ |
| `test_generate_cog.py` | 2 | Não | 4 | ✅ |
| `test_e2e_avg.py` | 3 | **Sim** | 1 | ✅ |
| `test_e2e_max.py` | 3 | **Sim** | 1 | ✅ |
| `test_e2e_maxx.py` | 3 | **Sim** | 1 | ✅ |
| `test_e2e_old_methods.py` | 3 | **Sim** | 3 | ✅ |
| **Total** | | | **54** | **✅ 54/54** |

---
## Resumo

| # | Arquivo | Bug | Impacto |
|---|---|---|---|
| 1 | `smosaic_merge_tifs.py` | `nodata=0` hardcoded ignorava `NaN` dos novos métodos | Buracos em mosaicos multi-tile |
| 2 | `smosaic_fix_baseline_number.py` | `nodata=NaN` inválido ao forçar `dtype=int16` | `ValueError` na execução com baseline > 400 |
| 3 | `smosaic_new_methods.py` | Bandas 20m lidas sem resample para resolução de referência | `IndexError` ao misturar B04 (10m) e B12 (20m) |

---
## Arquivos Não Modificados

Os seguintes arquivos da pipeline original **não tiveram o código-fonte alterado**. Testes de regressão foram adicionados para todos eles (seção 8).

```
smosaic_clip_raster.py          smosaic_generate_cog.py
smosaic_collection_get_data.py  smosaic_get_dataset_extents.py
smosaic_collection_query.py     
smosaic_count_pixels.py         
smosaic_download_stream.py      smosaic_reproject_tif.py
smosaic_filter_scenes.py        smosaic_spectral_indices.py
                                smosaic_utils.py
```

---
## 10. Execuções reais com dados STAC BDC

Após a integração completa dos novos métodos, a pipeline foi executada com dados reais da coleção `S2_L2A-1` para a **bbox de Mato Grosso** (`-57.016, -20.539, -55.961, -20.070`), período **2024-08-01 a 2024-08-30**. Os resultados estão nos diretórios `output_*/`.

### Visão geral das execuções

| Diretório | Método | Bandas solicitadas | `banda_ref` | Bandas baixadas | COGs gerados |
|---|---|---|---|---|---|
| `output_nbr/` | `min` | **NBR** | — | B08, B12 (via `COMPUTED_BANDS`) | NBR + PROVENANCE + SCL |
| `output_lcf/` | `lcf` | B02, B04, B03 | — | B02, B04, B03 | B02 + B04 + B03 + PROVENANCE + SCL |
| `output_minx/` | `minx` | B02, B04, B03 | B04 | B02, B04, B03 | B02 + B04 + B03 + PROVENANCE + SCL |
| `output_minx_queimadas/` | `minx` | B12, B08, B04 | B08 | B12, B08, B04 | B12 + B08 + B04 + PROVENANCE + SCL |
| `output_min/` | `min` | B12, B08, B04 | — | B12, B08, B04 | B12 + B08 + B04 + PROVENANCE + SCL |

> **Destaque — `output_nbr/`:** o usuário solicita `bands=['NBR']`, mas `COMPUTED_BANDS` expande para `['B08', 'B12']` antes do download. Após a reprojeção, `calcular_nbr_cubo` computa o índice em memória dentro de `compose_new_method`, e o arquivo final `S2_L2A-MT-NBR_20240801_20240830_COG.tif` é gravado em float32 com `nodata=NaN`. A proveniência indica o dia do ano (1–366) em que cada pixel atingiu o mínimo de NBR — proxy do evento de queimada mais intenso no período.

> **`output_minx_queimadas/`:** `minx` com `banda_ref='B08'` seleciona, para cada pixel, a data em que B08 (NIR) foi mais baixo — indicativo de área com menor vegetação/maior impacto. O trio B12/B08/B04 na mesma data gera uma falsa-cor da cena de queimada.

In [1]:
import os
from smosaic import mosaic

stac_url = "https://data.inpe.br/bdc/stac/v1"


### Execução 1 — `min` com `bands=['NBR']` → `output_nbr/`

In [2]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    bbox="-57.016296, -20.539257, -55.961609, -20.070443",
    projection_output=4326,
    output_dir=os.path.join("output_nbr"),
    mosaic_method="min",
    start_year=2024,
    start_month=8,
    start_day=1,
    end_year=2024,
    end_month=8,
    end_day=30,
    bands=["NBR"],
)

--- Starting parallel processing with 8 processes. ---

[Process 73380] Starting to process period: 2024-08-01 to 2024-08-30



Composing (min): 100%|██████████| 3/3 [12:33<00:00, 251.32s/it]


Raster saved to: output_nbr/S2_L2A-MT-NBR_20240801_20240830_COG.tif
Raster saved to: output_nbr/S2_L2A-MT_SCL_20240801_20240830_COG.tif
Raster saved to: output_nbr/S2_L2A-MT-PROVENANCE_20240801_20240830_COG.tif


In [ ]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    grid="BDC_SM_V2",
    tile_id="020019",
    projection_output="BDC",
    output_dir=os.path.join("output"),   
    mosaic_method="avg", 
    start_year=2026,
    start_month=1,
    start_day=1,
    duration_days=16, 
    bands=["B02","B04","B03"]
)

In [ ]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    grid="BDC_SM_V2",
    tile_id="020019",
    projection_output="BDC",
    output_dir=os.path.join("output"),
    mosaic_method="maxx",
    start_year=2026,
    start_month=1,
    start_day=1,
    duration_days=16,
    bands=["B02", "B04", "B03"],
    banda_ref="B04",
    k=1,
)


### Execução 2 — `lcf` com B02/B04/B03 → `output_lcf/`

In [4]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    bbox="-57.016296, -20.539257, -55.961609, -20.070443",
    projection_output=4326,
    output_dir=os.path.join("output_lcf"),
    mosaic_method="lcf",
    start_year=2024,
    start_month=8,
    start_day=1,
    end_year=2024,
    end_month=8,
    end_day=30,
    bands=["B02", "B04", "B03"],
)


Downloading... : 100%|██████████| 24/24 [19:19<00:00, 48.32s/ itens]

Successfully download 24 files to S2_L2A-1
--- Starting parallel processing with 8 processes. ---



[Process 11019] Starting to process period: 2024-08-01 to 2024-08-30



Processing B03...: 100%|██████████| 24/24 [01:44<00:00,  4.35s/it]


Raster saved to: output_lcf/S2_L2A-MT-B02_20240801_20240830_COG.tif
Raster saved to: output_lcf/S2_L2A-MT_SCL_20240801_20240830_COG.tif
Raster saved to: output_lcf/S2_L2A-MT-PROVENANCE_20240801_20240830_COG.tif
Raster saved to: output_lcf/S2_L2A-MT-B04_20240801_20240830_COG.tif
Raster saved to: output_lcf/S2_L2A-MT-B03_20240801_20240830_COG.tif


### Execução 3 — `minx` com B02/B04/B03, `banda_ref='B04'` → `output_minx/`

In [3]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    bbox="-57.016296, -20.539257, -55.961609, -20.070443",
    projection_output=4326,
    output_dir=os.path.join("output_minx"),
    mosaic_method="minx",
    start_year=2024,
    start_month=8,
    start_day=1,
    end_year=2024,
    end_month=8,
    end_day=30,
    bands=["B02", "B04", "B03"],
    banda_ref="B04",
    k=1,
)


--- Starting parallel processing with 8 processes. ---

[Process 57373] Starting to process period: 2024-08-01 to 2024-08-30



Composing (minx): 100%|██████████| 3/3 [21:50<00:00, 436.74s/it]


Raster saved to: output_minx/S2_L2A-MT-B02_20240801_20240830_COG.tif
Raster saved to: output_minx/S2_L2A-MT_SCL_20240801_20240830_COG.tif
Raster saved to: output_minx/S2_L2A-MT-PROVENANCE_20240801_20240830_COG.tif
Raster saved to: output_minx/S2_L2A-MT-B04_20240801_20240830_COG.tif
Raster saved to: output_minx/S2_L2A-MT-B03_20240801_20240830_COG.tif


### Execução 4 — `minx` com B12/B08/B04, `banda_ref='B08'` → `output_minx_queimadas/`

In [4]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    bbox="-57.016296, -20.539257, -55.961609, -20.070443",
    projection_output=4326,
    output_dir=os.path.join("output_minx_queimadas"),
    mosaic_method="minx",
    start_year=2024,
    start_month=8,
    start_day=1,
    end_year=2024,
    end_month=8,
    end_day=30,
    bands=["B12", "B08", "B04"],
    banda_ref="B08",
    k=1,
)

Downloading... : 100%|██████████| 24/24 [08:16<00:00, 20.70s/ itens]


Successfully download 24 files to S2_L2A-1
--- Starting parallel processing with 8 processes. ---

[Process 66608] Starting to process period: 2024-08-01 to 2024-08-30



Composing (minx): 100%|██████████| 3/3 [00:46<00:00, 15.48s/it]


Raster saved to: output_minx_queimadas/S2_L2A-MT-B12_20240801_20240830_COG.tif
Raster saved to: output_minx_queimadas/S2_L2A-MT_SCL_20240801_20240830_COG.tif
Raster saved to: output_minx_queimadas/S2_L2A-MT-PROVENANCE_20240801_20240830_COG.tif
Raster saved to: output_minx_queimadas/S2_L2A-MT-B08_20240801_20240830_COG.tif
Raster saved to: output_minx_queimadas/S2_L2A-MT-B04_20240801_20240830_COG.tif


### Execução 5 — `min` com B12/B08/B04 → `output_min/`

In [5]:
result = mosaic(
    name="MT",
    data_dir=os.path.abspath(""),
    stac_url=stac_url,
    collection="S2_L2A-1",
    bbox="-57.016296, -20.539257, -55.961609, -20.070443",
    projection_output=4326,
    output_dir=os.path.join("output_min"),
    mosaic_method="min",
    start_year=2024,
    start_month=8,
    start_day=1,
    end_year=2024,
    end_month=8,
    end_day=30,
    bands=["B12", "B08", "B04"],
)

--- Starting parallel processing with 8 processes. ---

[Process 67641] Starting to process period: 2024-08-01 to 2024-08-30



Composing (min): 100%|██████████| 3/3 [01:02<00:00, 20.95s/it]


Raster saved to: output_min/S2_L2A-MT-B12_20240801_20240830_COG.tif
Raster saved to: output_min/S2_L2A-MT_SCL_20240801_20240830_COG.tif
Raster saved to: output_min/S2_L2A-MT-PROVENANCE_20240801_20240830_COG.tif
Raster saved to: output_min/S2_L2A-MT-B08_20240801_20240830_COG.tif
Raster saved to: output_min/S2_L2A-MT-B04_20240801_20240830_COG.tif
